<a href="https://colab.research.google.com/github/ipeirotis/dealing_with_data/blob/master/01-Pandas/A4-NYPD_Vehicle_Collisions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Pandas

In [ ]:
# You need to change the project_id to your own Google Cloud project
project_id = "nyu-datasets" # <<<<<< CHANGE THIS

In [ ]:
# @title Setup and preliminaries

# This code snippet authenticates the user to access
# Google Cloud services from within Colab. This is
# necessary to interact with services like BigQuery.
# The auth.authenticate_user()  opens a browser window
# for the user to log in and authorize access.
!pip install -q google-cloud-bigquery

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()


client = bigquery.Client(project=project_id)


import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Render our plots with high resolution
%config InlineBackend.figure_format = 'retina'

# Make the graphs a bit bigger
matplotlib.style.use(["seaborn-v0_8-talk", "seaborn-v0_8-ticks", "seaborn-v0_8-whitegrid"])

# Setting the default figure size for Pandas plots
pd.options.plotting.backend = 'matplotlib'
plt.rcParams['figure.figsize'] = [10, 3]




In [ ]:
# @title Geospatial capabilities setup

# We install the geospatial libraries to be used for Task 10 (if desired)

import geopandas as gpd

# Dataset from NYC Open Data: https://data.cityofnewyork.us/City-Government/2020-Neighborhood-Tabulation-Areas-NTAs-/9nt8-h7nd/
df_nyc = gpd.GeoDataFrame.from_file('https://data.cityofnewyork.us/resource/9nt8-h7nd.geojson')

## Exercise: NYPD Vehicle Collisions

* We interacted with the NYC Restaurant Inspection Data. Now, let's download another dataset, and do some analysis. We will focus on the [NYPD Vehicle Collisions](https://data.cityofnewyork.us/Public-Safety/NYPD-Motor-Vehicle-Collisions/h9gi-nx95/data) data set.


### Task 1

Load the dataset for all the collisions after Jan 1st, 2020. We will need to load two tables, with the appropriate date restrictions:

* `nyu-datasets.collisions.collisions`
* `nyu-datasets.collisions.causes_types`


In [ ]:
# This query returns back the collisions table
# sql = '''
    #YOUR CODE HERE
# '''
# c = client.query(sql).to_dataframe()

In [ ]:
# This query returns back the causes_types table
# sql = '''
    #YOUR CODE HERE
# '''
#	v = client.query(sql).to_dataframe()

#### Solution

In [ ]:
# This query returns back the collisions table
sql = '''
	SELECT *
  FROM nyu-datasets.collisions.collisions
  WHERE DATE_TIME >= '2020-01-01'
'''

c = client.query(sql).to_dataframe()

In [ ]:
# This query returns back the causes_types table
sql = '''
	SELECT *
  FROM nyu-datasets.collisions.causes_types
  WHERE UNIQUE_KEY IN (
  	SELECT UNIQUE_KEY
    FROM nyu-datasets.collisions.collisions
    WHERE DATE_TIME >= '2020-01-01'
  )
'''

v = client.query(sql).to_dataframe()

### Task 2

Find out the most common contributing factors to the collisions, for all accidents after Jan-1-2020. You can either use the dataframe that we loaded above (`causes_types`)  or issue an SQL query and fetch a new dataframe.

#### Solution

In [ ]:
# Task 2: Find out the most common contributing factors to the collisions.
v['CAUSE'].value_counts() #.plot(kind='barh')

In [ ]:
# Task 2:
v['CAUSE'].value_counts().head(10).plot(kind='barh')

In [ ]:
# Notice the  difference if we use "COUNT(DISTINCT UNIQUE_KEY)"
# instead of COUNT(*). The former counts accidents, the later vehicles
factors_sql = '''
	SELECT CAUSE, COUNT(*) AS cnt, COUNT(DISTINCT UNIQUE_KEY) AS cnt_acc
  FROM nyu-datasets.collisions.causes_types
  WHERE UNIQUE_KEY IN (
  	SELECT UNIQUE_KEY
    FROM nyu-datasets.collisions.collisions
    WHERE DATE_TIME >= '2020-01-01'
  )
  GROUP BY CAUSE
  ORDER BY cnt DESC
'''

factors_df = client.query(factors_sql).to_dataframe()

factors_df.head(10)

In [ ]:
(
 factors_df
 .head(10) # keep the top-10 factors
 #.tail(9) # uncomment if you want to eliminate "UNSPECIFIED" (the top-1)
 .sort_values('cnt')
 .plot(
     kind='barh',
     x='CAUSE',
     y='cnt_acc',
     figsize=(8,4),
     xlabel="Number of accidents",
     ylabel="Contributing factor",
     title="Most common contributing factors to collisions"
  )
)

pass

### Task 3

Break down the number of collisions by borough.





#### Solution

In [ ]:
# Task 3: Break down the number of collisions by borough.
(
    c['REPORTED_BOROUGH']
    .value_counts()
    .plot(
        kind='barh',
        figsize=(6,2),
        xlabel="Number of collided vehicles",
        ylabel="Borough"
    )
)

In [ ]:
# Notice that you can remove the date time restriction and you
# will get the results back equally fast. If you try to load the
# whole collisions table in a dataframe, and then do the value_counts
# or a pivot table, it may take quite a while.

boro_sql = '''
	SELECT REPORTED_BOROUGH, COUNT(*) AS cnt
  FROM nyu-datasets.collisions.collisions
  WHERE DATE_TIME >= '2020-01-01'
  GROUP BY REPORTED_BOROUGH
  ORDER BY cnt DESC
'''

boro_df = client.query(boro_sql).to_dataframe()

(
    boro_df
    .plot(
        kind='barh',
        x='REPORTED_BOROUGH',
        y='cnt',
        figsize=(6,2),
        xlabel="Number of collided vehicles",
        ylabel="Borough"
    )
)


### Task 4

Find out the how many collisions had 0 persons injured, 1 persons injured, etc. persons injured in each accident. Use the `value_counts()` approach. You may also find the `.plot(logy=True)` option useful when you create the plot to make the y-axis logarigthmic.


#### Solution

In [ ]:
# "Chain" style of writing data maniputation operations
(
  c['PERSONS_INJURED'] # take the num of injuries column
  .value_counts() # compure the freuquency of each value
  .sort_index() # sort the results based on the index value instead of the frequency,
                # which is the default for value_counts
  .plot( # and plot the results
      kind='line', # we use a line plot because the x-axis is numeric/continuous
      marker='o',  # we use a marker to mark where we have data points
      logy=True, # make the y-axis logarithmic
      rot=0, # rotate the x-axis labels
      xlabel="Number of injuries",
      ylabel="Number of collisions",
      title="Analysis of number of injuries per collision"
  )
)
pass

In [ ]:
injuries_sql = '''
	SELECT PERSONS_INJURED, COUNT(*) AS cnt
  FROM nyu-datasets.collisions.collisions
  WHERE DATE_TIME >= '2020-01-01'
  GROUP BY PERSONS_INJURED
  ORDER BY cnt DESC
'''

injuries_df = client.query(injuries_sql).to_dataframe()

# "Chain" style of writing data maniputation operations
plot = (
    injuries_df # take the num of injuries column
    .sort_values('PERSONS_INJURED')
    .plot( # and plot the results
        x = 'PERSONS_INJURED',
        y = 'cnt',
        kind='line', # we use a line plot because the x-axis is numeric/continuous
        marker='o',  # we use a marker to mark where we have data points
        logy=True, # make the y-axis logarithmic
        xlabel="Number of injuries",
        ylabel="Number of collisions",
        title="Analysis of number of injuries per collision"
    )
)

pass

### Task 5

(a) Compute the average number of injuries and deaths per accident, broken down by borough. Use the `pivot_table` functionality, putting `BOROUGH` as the index. You can answer this query by generating two separate tables, or you can create a single table by using the fact that you can pass a list of attributes/columns to the `values` parameter of the pivot table.

(b) Repeat the exercise above, but break down the average number of deaths and injuries using the cause for the accident. (Do not worry that each accident may have multiple causes.) You will need to **join** the tables `collisions` and `vehicles_involves`; you can do the join either in SQL or in pandas, using the `pd.merge` command. Use the `sort_values` command to sort the results, putting on top the contributing factors that generate the highest number of deaths. Limit to the 10-deadliest causes.

#### Solution

In [ ]:
pd.pivot_table(
    data = c,
    index = 'REPORTED_BOROUGH',
    aggfunc = 'mean',
    values = ['PERSONS_INJURED', 'PERSONS_KILLED']
)

In [ ]:
# By keeping only the minimum attributes that we need, we speed up
# the execution, as we do not bring back data that we will discard anyway
sql = '''
  SELECT C.REPORTED_BOROUGH
        , AVG(C.PERSONS_INJURED) AS PERSONS_INJURED
        , AVG(C.PERSONS_KILLED) AS PERSONS_KILLED
  FROM nyu-datasets.collisions.collisions C
  WHERE DATE_TIME >= '2020-01-01'
  GROUP BY C.REPORTED_BOROUGH
'''
result = client.query(sql).to_dataframe()
display(result)

In [ ]:
# By keeping only the minimum attributes that we need, we speed up
# the execution, as we do not bring back data that we will discard anyway
sql = '''
  SELECT V.CAUSE
        , AVG(C.PERSONS_INJURED) AS PERSONS_INJURED
        , AVG(C.PERSONS_KILLED) AS PERSONS_KILLED
  FROM nyu-datasets.collisions.collisions C
    JOIN nyu-datasets.collisions.causes_types V ON C.UNIQUE_KEY = V.UNIQUE_KEY
  WHERE DATE_TIME >= '2020-01-01'
  GROUP BY V.CAUSE
'''

result = client.query(sql).to_dataframe()

In [ ]:
(
  result
 .set_index('CAUSE')
 .sort_values('PERSONS_KILLED',ascending=False)
 .head(20)

)
#

### Task 6

Break down the number of accidents by borough and cause. Use the `pivot_table` function of Pandas, making the values of "borough" to be  columns and cause to be rows.


#### Solution

In [ ]:
# By keeping only the minimum attributes that we need, we speed up
# the execution, as we do not bring back data that we will discard anyway
sql = '''
  SELECT C.REPORTED_BOROUGH,  V.CAUSE, COUNT(DISTINCT C.UNIQUE_KEY) AS cnt
  FROM nyu-datasets.collisions.collisions C
    JOIN nyu-datasets.collisions.causes_types V ON C.UNIQUE_KEY = V.UNIQUE_KEY
  WHERE DATE_TIME >= '2020-01-01'
  GROUP BY C.REPORTED_BOROUGH,  V.CAUSE
'''

result = client.query(sql).to_dataframe()

In [ ]:
pivot = pd.pivot_table(
    data = result, # we analyze the df (accidents) dataframe
    index = 'CAUSE',
    columns = 'REPORTED_BOROUGH',
    values = 'cnt',
    aggfunc = 'sum'
)

# Create an extra column showing the total deaths across boroughs (=columns)
pivot["Total"] = pivot.sum(axis="columns")

# Sort the dataframe by descending order of the values in the column "Total"
pivot = pivot.sort_values("Total", ascending=False)

pivot.head(10)

### Task 7

Find the dates with the most accidents. Can you figure out what happened on these days?


#### Solution

In [ ]:
sql = '''
  SELECT DATE(DATE_TIME) AS accident_date, COUNT(*) AS cnt
  FROM nyu-datasets.collisions.collisions
  GROUP BY DATE(DATE_TIME)
  ORDER BY cnt DESC
'''

date_df = client.query(sql).to_dataframe()

In [ ]:
date_df

In [ ]:
(
  pd.pivot_table(
      data = date_df,
      index = 'accident_date',
      values = 'cnt',
  )
  # .resample('1D').sum()
  .sort_values('cnt', ascending=False)
)

### Task 8

Plot the number of accidents per day. Try to eliminate the effects of seasonality by resampling and calculating values on a weekly or monthly basis (Hint: Ensure that your date column is in the right datatype and that it is properly sorted, before attempting a `resample`)


#### Solution

In [ ]:
sql = '''
  SELECT DATE(DATE_TIME) AS accident_date,
      COUNT(*) AS cnt,
      SUM(PERSONS_INJURED) AS persons_injured,
      SUM(PERSONS_KILLED) AS persons_killed
  FROM nyu-datasets.collisions.collisions
  GROUP BY DATE(DATE_TIME)
  ORDER BY cnt DESC
'''

date_df = client.query(sql).to_dataframe()

(
  pd.pivot_table(
      data = date_df,
      index = 'accident_date',
      values = 'persons_injured',
  )
  # .resample('1W') # take periods of 1 week
  # .sum() # sum the number of accidents per period
  .plot(figsize=(15,3)) # plot the result
)



In [ ]:
# Convert the 'accident_date' from 'object' to datetime
date_df['date'] = pd.to_datetime(date_df['accident_date'])

(
  pd.pivot_table(
      data = date_df,
      index = 'date',
      values = 'cnt',
  )
  .resample('1W') # take periods of 1 week
  .sum() # sum the number of accidents per period
  .plot(figsize=(15,3)) # plot the result
)

### Task 9

We want to analyze the timing patterns of accidents that lead to death or injury.

We will do the analysis by creating histograms showing the frequency of deadly vs non-deadly accidents throughout the day. By comparing the two histograms we will be able to understand if time of day is correlated with deadly accidents or not.

Steps to follow:
* Create an `HOUR` column that captures the hour of the day that the accident happened.
* Create a boolean column `DEATH` that is true when someone was killed in the accident (i.e., `NUMBER OF PERSONS KILLED > 0`).
* Create a boolean column `INJURY` that is true when someone was injured in the accident (i.e., `NUMBER OF PERSONS INJURED > 0`).
* Query the dataframe to get back the deadly accidents and create a histogram of deadly accidents over time. Do the same for non-deadly accidents.
* To allow a more direct visual comparison of the two histograms, we want to merge them in one plot. Since the number of accidents without deaths is *much* higher, we want the histograms to be normalized (i.e., `density=True`).
* It is also a good idea to make the histographs partially transparent, to allow for easier comparison of the two histograms.


#### Solution

In [ ]:
sql = '''
  SELECT UNIQUE_KEY, DATE_TIME
        , EXTRACT(HOUR FROM DATE_TIME) AS HOUR
        , PERSONS_INJURED>0 AS INJURY
        , PERSONS_KILLED>0 AS DEATH
  FROM nyu-datasets.collisions.collisions
'''

df = client.query(sql).to_dataframe()


In [ ]:
pd.pivot_table(
    data = df,
    index = 'HOUR',
    values = ['INJURY','DEATH'],
    aggfunc = 'mean',
).plot(
    secondary_y = ['DEATH'],
    title = "Probability an accident leads to death or injury",
    xlabel= "Hour of the day",
    ylabel= "Probability"
)

##### And let's do the same analysis over time

In [ ]:
pd.pivot_table(
    data = df,
    index = 'DATE_TIME',
    values = 'INJURY',
    aggfunc = 'mean',
).resample('1ME').mean().plot()

In [ ]:
pd.pivot_table(
    data = df,
    index = 'DATE_TIME',
    values = 'DEATH',
    aggfunc = 'mean',
).resample('1ME').mean().plot()

In [ ]:
import seaborn as sns

In [ ]:
sns.kdeplot(data = df, x ='HOUR', hue='DEATH', common_norm=False, bw_adjust=2, cut=0)

In [ ]:
sns.kdeplot(data = df, x ='HOUR', hue='INJURY', common_norm=False, bw_adjust=2, cut=0)

### Task 10

Create a plot that shows the locations of the cyclist deaths. Filter first for accidents where there was a cyclist fatality, and then use a scatterplot on longitude and latitude. In the next step, create a 2-d kernel density plot to show the same information.

#### Solution

In [ ]:
sql = "SELECT LONGITUDE, LATITUDE FROM nyu-datasets.collisions.collisions WHERE CYCLISTS_KILLED > 0"

cyclist_dead = client.query(sql).to_dataframe()

In [ ]:
(
    cyclist_dead
    .plot(
        kind='scatter',
        x='LONGITUDE',
        y='LATITUDE',
        # s=1,
        figsize=(10,10)
    )
)

In [ ]:
scatter = (
    cyclist_dead
    .plot(
        kind='scatter',
        x='LONGITUDE',
        y='LATITUDE',
        figsize=(10,10)
    )
)

sns.kdeplot(
    data = cyclist_dead,
    x='LONGITUDE',
    y='LATITUDE',
    shade=True,
    gridsize=100,
    cmap='rainbow',
    alpha=0.75,
    n_levels=50,
    ax=scatter
)

In [ ]:
base = df_nyc.plot(
    linewidth=0.5,
    color='White',
    edgecolor='Black',
    figsize=(10, 10),
    alpha=0.75
)

scatter = (
    cyclist_dead
    .plot(
        kind='scatter',
        x='LONGITUDE',
        y='LATITUDE',
        ax = base # plot it on top of the NYC boundaries
    )
)

sns.kdeplot(
    data = cyclist_dead,
    x='LONGITUDE',
    y='LATITUDE',
    fill=True, # Whether to color between the levels (True) or just keep the contours
    gridsize=100, # The resoltion of the 2d plot
    cmap='rainbow', # Color scheme
    alpha=0.5, # make the 2d density plot a bit transparent
    n_levels=20, # calculate 20 levels for the 2d density plot
    ax=scatter # plot it on top of the scatter plot
)

# Congestion Pricing and Accidents Dataset

* You are asked to analyze the effect of congestion pricing, which started on January 5th, 2025 using this dataset. How do you approach the problem?